In [ ]:
%pip install --upgrade --quiet ipywidgets langchain-community langchain langchain-openai faiss-cpu beautifulsoup4

### Define variables

In [28]:
import os

LLM_MODEL = 'gpt-4o'
TEMPERATURE = 0
K = 10
EMBEDDINGS_MODEL = 'text-embedding-ada-002'

OPENAI_API_KEY = os.getenv('OPENAPI_API_KEY')

### Define embeddings

In [29]:
from langchain_openai import OpenAIEmbeddings
EMBEDDINGS = OpenAIEmbeddings( model=EMBEDDINGS_MODEL, openai_api_key=OPENAI_API_KEY )

### Define LLM

In [30]:
from langchain_openai import ChatOpenAI
LLM = ChatOpenAI(model=LLM_MODEL, openai_api_key=OPENAI_API_KEY, temperature=TEMPERATURE)

### Utilities

In [31]:
# Document Converters

import pathlib
import pymupdf4llm

def pdf_to_md(path):
    md_text = pymupdf4llm.to_markdown(path)
    output_path = pathlib.Path(path).with_suffix('.md')
    pathlib.Path(output_path).write_bytes(md_text.encode())


# Document loaders

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

def load_pdf(path, vector_store):
    loader = PyPDFLoader(path)
    docs = []
    for page in loader.load():
        docs.append(page)
    vector_store.add_documents(documents=docs)
    return docs

def load_pdf(path):
  loader = PyPDFLoader(path)
  pages = loader.load()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
  docs = text_splitter.split_documents(pages)
  return docs

def load_markdown(path, vector_store):
    markdown = open(path, 'r').read()
    docs = MarkdownHeaderTextSplitter(
        headers_to_split_on = [ ('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3') ], 
        strip_headers=False
    ).split_text(markdown)
    vector_store.add_documents(documents=docs)
    return docs

# Document printers
import json

def print_input_docs(docs):
    for doc in docs:
        print(doc.page_content)
        print ('\n' + '-'*80 + '\n')

def print_output_docs(docs):
    for doc in docs:
        print (doc.page_content.replace('\n', ' ') + '\n')
        print (json.dumps(doc.metadata))
        print ('\n' + '-'*80 + '\n')

### Create in-memory vector store

In [39]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(EMBEDDINGS)
retriever = vector_store.as_retriever(search_kwargs={'k': K})

### Load Documents

In [40]:
# pdf_to_md('events/2025JanFebEventCalendar.pdf')
print_input_docs(load_markdown('events/2025JanFebEventCalendar-updated.md', vector_store))

# Upcoming Events  
Tickets for all Lifestyle Services events are available online at  www.SunCityHiltonHead.org  
General admission events and  Magnolia Hall shows are limited  to four tickets per household. Pinckney Hall seated events are limited to eight tickets per  household unless otherwise stated.  
Only credit or debit cards  accepted for online purchases.  CAM card charges only accepted  at Palmetto Commons.  
No computer? Buy your tickets  and use your activity card for purchases at Lifestyle Services.  
Print your tickets at home or at Lifestyle Services before the event.  
Online ticketing will close one  hour prior to each event. Walk-ins permitted depending on space. Pay with cash or activity card.  
No refunds will be issued for tickets purchased.

--------------------------------------------------------------------------------

## Events  
### January 2025  
Thursday, January 2
**Ticket Sales**
Tickets for all February events go on sale
online and at Lifestyle Services.

### Query and print retrieved docs

In [42]:
question = 'list the upcomoing events'

print_output_docs(retriever.invoke(question))

# Upcoming Events   Tickets for all Lifestyle Services events are available online at  www.SunCityHiltonHead.org   General admission events and  Magnolia Hall shows are limited  to four tickets per household. Pinckney Hall seated events are limited to eight tickets per  household unless otherwise stated.   Only credit or debit cards  accepted for online purchases.  CAM card charges only accepted  at Palmetto Commons.   No computer? Buy your tickets  and use your activity card for purchases at Lifestyle Services.   Print your tickets at home or at Lifestyle Services before the event.   Online ticketing will close one  hour prior to each event. Walk-ins permitted depending on space. Pay with cash or activity card.   No refunds will be issued for tickets purchased.

{"Header 1": "Upcoming Events"}

--------------------------------------------------------------------------------

## Events   ### January 2025   Thursday, January 2 **Ticket Sales** Tickets for all February events go on sale 

In [41]:
from langchain.chains import RetrievalQA 
    
print(RetrievalQA.from_chain_type( llm=LLM, chain_type='stuff', retriever=retriever ).invoke(question)['result'])


Here are the upcoming events:

### January 2025
- **Thursday, January 2**
  - Ticket Sales for February events
  - Trivia Night (Sold out)

- **Monday, January 6**
  - Irish Coffee Night (Free event)
  - Cold Spring Harbor Band (Sold out)

- **Tuesday, January 7**
  - Irish Coffee Night (Free event)
  - Cold Spring Harbor Band (Sold out)

- **Thursday, January 9**
  - Make it & Take it: Scalloped Shell Beaded Necklace

- **Friday, January 10**
  - Meet the Author: Margie Hamner (Free event)

- **Monday, January 13**
  - Paint & Pinot: Brilliant Pelican
  - Movie Night: _Twisters_ (Free event)

- **Wednesday, January 15**
  - Chartered Club Annual Meeting
  - Funny Business: Murray Valeriano (Sold out)

- **Thursday, January 16**
  - Trivia Night

- **Tuesday, January 21**
  - Irish Coffee Night (Free event)
  - Boilermaker Jazz Band

- **Wednesday, January 22**
  - Irish Coffee Night (Free event)
  - Boilermaker Jazz Band

- **Thursday, January 23**
  - Bourbon & Blues: Mac Arnold

- *